# Sliding-window conformal intervals

The point forecast is the frozen LightGBM from the training window. Conformity scores are out-of-sample residuals. Training residuals are not used. The calibration period chooses the window. The test period is scored once.

This is an inductive sliding-window residual method in the sense of EnbPI (Xu and Xie, 2021; journal version 2023, arXiv:2010.09107), not a full transductive conformal predictive system. A full CPS would recompute a conformity score for every candidate price. Here the predictive distribution at an origin is the point forecast plus the empirical distribution of residuals in a recent window. A hard window is also a special case of weighted conformal prediction beyond exchangeability (Barber, Candès, Ramdas and Tibshirani, 2023, Annals of Statistics, arXiv:2202.13415).

## How wide should the intervals be?

The useful width is not a single number of dollars. It is the smallest width that still meets a stated coverage target on data the model was not tuned on. Four results from the electricity and conformal literature fix the design.

Kath and Ziel (2021, International Journal of Forecasting, arXiv:1905.07886) showed that conformal intervals can be sharp and reliable in day-ahead and intraday power markets, and that the scale component is what stops one spike from setting the width of every later hour. Their intraday intervals are tighter than day-ahead intervals because the point-forecast errors are smaller. A 5-minute horizon should have small typical errors, but WEM MCP still runs from about -1000 to 1100 $/MWh with excess kurtosis around 34. An unnormalized 95% residual quantile that has seen a cap event is therefore hundreds of dollars wide, while the median absolute error can be only a few dollars. A constant-width split-conformal interval is the wrong object for this market.

Xu and Xie require the window to be long enough for the quantile to settle, and short enough to track regime changes. For a two-sided 95% interval the tail probability on each side is 2.5%. A rough stability rule is at least about 30 expected tail residuals, so `0.025 * W >= 30`, or `W >= 1200` steps. At 5-minute frequency that is about 4 days. The candidate windows therefore start at 2 days (noisy at 95%, usable at 50-80%) and run through 7, 14 and 30 days. A 30-day window stays wide for a month after one cap event enters it. A 2-day window recovers faster and will undercover the next unseen spike.

Gibbs and Candès (2021, arXiv:2106.00170) target long-run coverage under arbitrary shift by updating the miscoverage level online. That is a different adaptation from a sliding residual window: ACI widens after misses, and the learning rate controls how violently the width moves. Zaffran et al. (2022, ICML, arXiv:2202.07282) apply that idea to electricity prices and show that a poorly chosen learning rate makes the intervals inefficient. This notebook does not implement ACI. The requested method is the sliding window. ACI remains the alternative if empirical coverage drifts and a fixed window cannot track it.

Locally adaptive scores follow the same idea as Kath and Ziel and as Lei et al. (2018, JASA): divide the residual by a scale known at the origin. The scale here is the median absolute residual over the previous day, lagged by one interval, with a floor of 1 $/MWh so a run of identical prices does not collapse the interval to zero. The interval is then `yhat + sigma_t * [q_lo, q_hi]`. In calm hours this should be a small multiple of recent typical error. After a volatile day it widens. It still cannot cover a spike that has not yet appeared in the window.

No method in this notebook has an unconditional finite-sample coverage guarantee. Exchangeability fails, and the forecasts are serially dependent. Barber et al. bound the coverage gap by distribution drift; they do not restore exact `1 - alpha` coverage. Empirical coverage is reported through time and on tail prices. QRA is not implemented, so this notebook does not claim a QRA-versus-CPS comparison.

## Algorithm

1. Fit LightGBM on the training window only. Freeze it.
2. Predict the calibration and test origins. Residuals are `MCP - yhat`.
3. For a candidate window `W` in {576, 2016, 4032, 8640} steps and a nominal level `1 - alpha`, the interval at origin `t` uses residuals whose targets fall in the previous `W` model-ready origins, excluding `t`.
4. With `n = W` finite scores, the lower rank is `floor((n+1) alpha/2)` and the upper rank is `ceil((n+1)(1-alpha/2))`.
5. Absolute method: add those residual quantiles to `yhat`. Normalized method: form the quantiles on `residual / sigma`, then multiply by the current lagged scale.
6. Score every candidate on the calibration origins that have a full 30-day residual window and a further day of scale history before that window, so absolute and normalized rows share the same origins. Keep a window only if 90% coverage is at least 0.89 and 95% coverage is at least 0.93. Among eligible windows, choose the narrowest mean 90% interval. If none are eligible, keep the highest 90% coverage.
7. Choose between absolute and normalized on that same calibration table. Apply the frozen choice, and the other reported variants, to the test period once.
8. Also report a fixed split-conformal interval that uses the whole calibration residual set. That is the exchangeable benchmark. It should be wider in calm hours and slower to adapt.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_root = Path.cwd().resolve()
if not (_root / "scripts").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import joblib

from scripts.conformal import ALPHAS, apply_fixed, apply_sliding, choose_primary, selection_table, sliding_crps
from scripts.experiment import MODEL_PATH, load_ready
from scripts.forecast_design import TARGET, TREE_FEATURES, slice_split
from scripts.metrics import interval_group_scores, interval_scores, pinball_loss
from scripts.point_models import persistence, predict_lgbm, select_lgbm

ready = load_ready()
train = slice_split(ready, "train")
calibration = slice_split(ready, "calibration")
test = slice_split(ready, "test")
if MODEL_PATH.exists():
    saved = joblib.load(MODEL_PATH)
    bundle = saved["bundle"]
    features = saved["features"]
else:
    bundle = select_lgbm(train, TREE_FEATURES, TARGET)
    features = TREE_FEATURES
yhat_cal = predict_lgbm(bundle, calibration, features)
yhat_test = predict_lgbm(bundle, test, features)
resid_cal = calibration[TARGET] - yhat_cal
print("calibration residuals", len(resid_cal))

In [ ]:
selection = selection_table(resid_cal, yhat_cal, calibration[TARGET])
selection

In [ ]:
primary = choose_primary(selection)
window = int(primary["window_steps"])
method = str(primary["method"])
print("frozen from calibration:", method, window)

history = pd.concat([
    calibration[TARGET] - yhat_cal,
    test[TARGET] - yhat_test,
]).sort_index()
rows = []
for alpha in ALPHAS:
    interval = apply_sliding(history, yhat_test, window, alpha, method)
    scores = interval_scores(test[TARGET], interval["lower"], interval["upper"])
    scores["nominal"] = 1 - alpha
    scores["pinball_lower"] = pinball_loss(test[TARGET], interval["lower"], alpha / 2)
    scores["pinball_upper"] = pinball_loss(test[TARGET], interval["upper"], 1 - alpha / 2)
    rows.append(scores)
fixed = apply_fixed(resid_cal.to_numpy(), yhat_test, 0.10)
fixed_scores = interval_scores(test[TARGET], fixed["lower"], fixed["upper"])
fixed_scores["nominal"] = 0.90
fixed_scores["method"] = "fixed_split"
sliding = pd.DataFrame(rows)
print("mean CRPS", float(sliding_crps(history, yhat_test, test[TARGET], window, method).mean()))
display(sliding)
fixed_scores

In [ ]:
persist_cal = persistence(calibration, "mcp_lag_5min")
persist_test = persistence(test, "mcp_lag_5min")
persist_history = pd.concat([
    calibration[TARGET] - persist_cal,
    test[TARGET] - persist_test,
]).sort_index()
persist_interval = apply_sliding(persist_history, persist_test, window, 0.10, method)
lgb_interval = apply_sliding(history, yhat_test, window, 0.10, method)
comparison = pd.DataFrame([
    {"base": "lightgbm", **interval_scores(test[TARGET], lgb_interval["lower"], lgb_interval["upper"])},
    {"base": "persistence_5min", **interval_scores(test[TARGET], persist_interval["lower"], persist_interval["upper"])},
])
lgb_crps = sliding_crps(history, yhat_test, test[TARGET], window, method)
persist_crps = sliding_crps(persist_history, persist_test, test[TARGET], window, method)
comparison["crps"] = [float(lgb_crps.mean()), float(persist_crps.mean())]
comparison

In [ ]:
spike_cut = float(train[TARGET].quantile(0.95))
floor_cut = float(train[TARGET].quantile(0.05))
tail = (test[TARGET] >= spike_cut) | (test[TARGET] <= floor_cut)
labels = pd.Series(np.where(tail.to_numpy(), "tail", "body"), index=test.index)
print(f"tail cuts from training only: {floor_cut:.2f}, {spike_cut:.2f}; test tail share {tail.mean():.3%}")
regime = pd.concat(
    [
        interval_group_scores(test[TARGET], lgb_interval["lower"], lgb_interval["upper"], labels, 0.10, lgb_crps).assign(base="lightgbm"),
        interval_group_scores(test[TARGET], persist_interval["lower"], persist_interval["upper"], labels, 0.10, persist_crps).assign(base="persistence_5min"),
    ],
    ignore_index=True,
)
months = pd.concat(
    [
        interval_group_scores(
            test[TARGET], lgb_interval["lower"], lgb_interval["upper"], test.index.to_period("M").astype(str), 0.10, lgb_crps
        ).assign(base="lightgbm"),
        interval_group_scores(
            test[TARGET], persist_interval["lower"], persist_interval["upper"], test.index.to_period("M").astype(str), 0.10, persist_crps
        ).assign(base="persistence_5min"),
    ],
    ignore_index=True,
)
display(regime)
months

## What can be claimed

Report empirical coverage, mean and median width, pinball loss and CRPS on the test origins. Also split those scores by month and by tail prices, using the training 5th and 95th percentiles of MCP as the tail cut. Do not describe the intervals as unconditionally valid at the nominal level. Do not tune the window, the scale floor, or the LightGBM parameters on the test table.

QRA is still absent. A later QRA comparison has to use this same ex-ante information set, the same origins, and the same horizon. `mapie` being listed in `requirements.txt` is not that comparison.

If 90% test coverage is materially below nominal, the next adaptation is ACI or a shorter window chosen again on calibration, not a threshold fit to the test misses. If coverage is nominal but the median width is many times the MAE, the interval is valid and still too blunt for a 5-minute decision; that is the spike-driven width Kath and Ziel warn about, and it is a reason to prefer the normalized score rather than to shrink the quantile on the test set.